In [ ]:
pip install datasets scikit-learn transformers torch accelerate evaluate

In [ ]:
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

In [ ]:
from datasets import load_dataset

ds = load_dataset("fancyzhx/ag_news")
print(ds)                 # train: 120,000 | test: 7,600
print(ds["train"][0])     # {'text': '...', 'label': 2}

label_names = ["World", "Sports", "Business", "Sci/Tech"]

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

train, test = ds["train"], ds["test"]

model = make_pipeline(
    TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=100000),
    LogisticRegression(max_iter=1000),
)
model.fit(train["text"], train["label"])

preds = model.predict(test["text"])
print(classification_report(test["label"], preds, target_names=label_names))

              precision    recall  f1-score   support

       World       0.93      0.91      0.92      1900
      Sports       0.96      0.98      0.97      1900
    Business       0.90      0.89      0.89      1900
    Sci/Tech       0.90      0.90      0.90      1900

    accuracy                           0.92      7600
   macro avg       0.92      0.92      0.92      7600
weighted avg       0.92      0.92      0.92      7600



In [ ]:
import torch
print(torch.cuda.is_available())
!nvidia-smi

True
Sun Nov 22 09:44:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [ ]:
import numpy as np, evaluate
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

ckpt = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(ckpt)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=128)

tokenized = ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    ckpt, num_labels=4,
    id2label=dict(enumerate(label_names)),
    label2id={n: i for i, n in enumerate(label_names)},
)

acc = evaluate.load("accuracy")
def metrics(p):
    return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

args = TrainingArguments(
    output_dir="agnews-distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics,
)
trainer.train()
trainer.save_model("agnews-distilbert/final")
tok.save_pretrained("agnews-distilbert/final")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('agnews-distilbert/final/tokenizer_config.json',
 'agnews-distilbert/final/tokenizer.json')

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model="agnews-distilbert/final")
print(clf("Stocks rally as the central bank holds interest rates steady"))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Business', 'score': 0.9718066453933716}]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import joblib
from google.colab import files

train = ds["train"]
model = make_pipeline(
    TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=100000),
    LogisticRegression(max_iter=1000),
)
model.fit(train["text"], train["label"])

joblib.dump(model, "news_classifier.pkl")
files.download("news_classifier.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r agnews-distilbert.zip agnews-distilbert/final
from google.colab import files
files.download("agnews-distilbert.zip")

  adding: agnews-distilbert/final/ (stored 0%)
  adding: agnews-distilbert/final/model.safetensors (deflated 8%)
  adding: agnews-distilbert/final/training_args.bin (deflated 53%)
  adding: agnews-distilbert/final/config.json (deflated 51%)
  adding: agnews-distilbert/final/tokenizer_config.json (deflated 43%)
  adding: agnews-distilbert/final/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp agnews-distilbert.zip /content/drive/MyDrive/

Mounted at /content/drive
